<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 02. U-Net — U-Net: Convolutional Networks for Biomedical Image Segmentation

- **원 논문:** [U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/pdf/1505.04597)
- **저자 / 발표:** Olaf Ronneberger, Philipp Fischer, and Thomas Brox · MICCAI 2015 (2015)
- **난이도 / 예상 시간:** 중급 · 약 95분
- **선수 지식:** encoder-decoder CNN, ConvTranspose2d, pixel-wise binary loss

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

U자형 contracting/expanding path와 같은 해상도 encoder feature의 concatenation을 2-level 모델로 구현한다. Fig. 1의 channel/shape 흐름, Eq. (3)의 경계 가중치 형태, pixel logits의 shape를 검증하고 합성 인접 사각형 분할에서 loss와 IoU를 측정한다.

**원 논문과 다른 것**

원 논문은 572×572 입력, valid 3×3 convolution 때문에 발생하는 crop-and-copy, 64→1024 channels, 탄성 변형 augmentation, 세포별 거리변환 weight map을 사용한다. 여기서는 32×32 입력, padding=1, 4→16 channels, 합성 binary masks를 사용한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2 및 Fig. 1 왼쪽: repeated 3×3 conv + ReLU와 2×2 max pool | `DoubleConv`, `enc1`, `enc2`, `pool` | encoder feature shape가 32²→16²→8²인지 assert |
| §2 및 Fig. 1 오른쪽: 2×2 up-convolution과 cropped feature copy/concat | `up2/up1`와 `torch.cat` skip connections | decoder concat channel 수와 최종 32×32 logits 검증 |
| §2, Eq. (1): pixel-wise softmax와 cross-entropy | binary인 미니 과제의 `binary_cross_entropy_with_logits` | logits/target 동일 shape와 유한 loss를 확인 |
| §2, Eq. (3): w(x)=wc(x)+w0 exp(-(d1+d2)^2/(2σ^2)) | `paper_border_weight`에서 두 object boundary 거리 사용 | 객체 사이 gap의 weight가 먼 corner보다 큰지 assert하고 heatmap 표시 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
d_l=\operatorname{Conv}\!\left(
\operatorname{Concat}(\operatorname{Up}(d_{l+1}),e_l)\right)
$$

- $e_l$은 encoder의 고해상도 feature, $d_l$은 decoder feature입니다.
- skip connection은 downsampling에서 잃을 수 있는 위치 정보를 decoder에 직접 전달합니다.
- encoder, bottleneck, up block을 class로 분리하고 concat 전후 channel을 assertion으로 검사합니다.
- 입력과 mask는 $[B,1,H,W]$이며 출력 logit도 같은 공간 shape을 가져야 합니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2 및 Fig. 1 왼쪽: repeated 3×3 conv + ReLU와 2×2 max pool
- **노트북 구현:** `DoubleConv`, `enc1`, `enc2`, `pool`
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** encoder feature shape가 32²→16²→8²인지 assert를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 원 논문은 572×572 입력, valid 3×3 convolution 때문에 발생하는 crop-and-copy, 64→1024 channels, 탄성 변형 augmentation, 세포별 거리변환 weight map을 사용한다. 여기서는 32×32 입력, padding=1, 4→16 channels, 합성 binary masks를 사용한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. U의 두 팔과 skip connection

Fig. 1의 왼쪽은 위치 정보를 압축하며 문맥을 얻고, 오른쪽은 해상도를 복원합니다.
핵심은 같은 scale의 왼쪽 feature를 오른쪽에 **channel 방향으로 복사·연결**하는 것입니다.
원 논문은 valid convolution 때문에 왼쪽 feature를 crop하지만, padding을 쓰는 이 미니
모델에서는 두 공간 크기가 이미 같습니다. 이는 의도적인 구현 차이입니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** image/mask [B,1,H,W] → skip features [B,C,H',W'] → logits [B,1,H,W]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(2)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def make_square_segments(samples=8, size=32):
    generator = torch.Generator().manual_seed(303)
    images = torch.zeros(samples, 1, size, size)
    masks = torch.zeros_like(images)
    for index in range(samples):
        top = 5 + index % 4
        left = 3 + index % 3
        masks[index, :, top : top + 11, left : left + 9] = 1
        masks[index, :, top + 1 : top + 12, left + 11 : left + 20] = 1
    blurred = F.avg_pool2d(masks, kernel_size=5, stride=1, padding=2)
    noise = 0.04 * torch.randn(images.shape, generator=generator)
    images = (0.75 * masks + 0.35 * blurred + noise).clamp(0, 1)
    return images, masks


unet_x, unet_y = make_square_segments()
unet_x, unet_y = ACCELERATOR.move(unet_x, unet_y)
assert unet_x.shape == unet_y.shape == (8, 1, 32, 32)
assert set(unet_y.unique().tolist()) == {0.0, 1.0}
print(ACCELERATOR.summary())
print("segmentation batch:", tuple(unet_x.shape))

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2 및 Fig. 1 왼쪽: repeated 3×3 conv + ReLU와 2×2 max pool / §2 및 Fig. 1 오른쪽: 2×2 up-convolution과 cropped feature copy/concat
- **이 셀의 책임:** `DoubleConv`, `enc1`, `enc2`, `pool` / `up2/up1`와 `torch.cat` skip connections
- **Tensor shape 계약:** image/mask [B,1,H,W] → skip features [B,C,H',W'] → logits [B,1,H,W]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** encoder feature shape가 32²→16²→8²인지 assert / decoder concat channel 수와 최종 32×32 logits 검증. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: (3×3 Conv → ReLU)를 두 번 수행하는 DoubleConv와
class DoubleConv(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, in_channels, out_channels):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


class UNetMini(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x, return_shapes=False):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


unet = UNetMini().to(DEVICE)


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 논문 주장 확인 실험

- **원문 위치:** §2 및 Fig. 1 왼쪽: repeated 3×3 conv + ReLU와 2×2 max pool / §2 및 Fig. 1 오른쪽: 2×2 up-convolution과 cropped feature copy/concat
- **이 셀의 책임:** `DoubleConv`, `enc1`, `enc2`, `pool` / `up2/up1`와 `torch.cat` skip connections
- **Tensor shape 계약:** image/mask [B,1,H,W] → skip features [B,C,H',W'] → logits [B,1,H,W]
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** encoder feature shape가 32²→16²→8²인지 assert / decoder concat channel 수와 최종 32×32 logits 검증. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: return_shapes=True로 Fig. 1의 축소/확대 shape와 최종 mask shape를 검증하세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §2, Eq. (3): w(x)=wc(x)+w0 exp(-(d1+d2)^2/(2σ^2))
- **이 셀의 책임:** `paper_border_weight`에서 두 object boundary 거리 사용
- **Tensor shape 계약:** image/mask [B,1,H,W] → skip features [B,C,H',W'] → logits [B,1,H,W]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 객체 사이 gap의 weight가 먼 corner보다 큰지 assert하고 heatmap 표시. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: 두 instance mask의 boundary까지 거리 d1,d2를 구하고 Eq. (3)을 반환하세요.
def _binary_boundary(mask):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def paper_border_weight(instances, w0=10.0, sigma=5.0):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 6. Loss의 대응 관계

원 논문 Eq. (1)은 두 class의 pixel-wise softmax입니다. binary foreground/background에서는
한 개 logit에 대한 `binary_cross_entropy_with_logits`가 같은 log-likelihood를 표현합니다.
아래 학습은 architecture를 빠르게 검증하면서 foreground 희소성을 보정하도록 binary
`pos_weight`를 사용합니다. 이는 Eq. (2)의 class-balancing `w_c`에 대응합니다. 바로 앞
셀의 Eq. (3) map을 loss의 `weight=` 인자로 넣으면 경계 강조 실험을 확장할 수 있습니다.

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §2, Eq. (1): pixel-wise softmax와 cross-entropy
- **이 셀의 책임:** binary인 미니 과제의 `binary_cross_entropy_with_logits`
- **Tensor shape 계약:** image/mask [B,1,H,W] → skip features [B,C,H',W'] → logits [B,1,H,W]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** logits/target 동일 shape와 유한 loss를 확인. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: class-balancing pos_weight를 둔 BCEWithLogits로 18 step 학습한 뒤 pixel IoU를 계산하세요.
# target과 sigmoid prediction을 나란히 시각화하고 loss 감소를 assert하세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.